# Download TissDiss h5 files
Downloads the unflagged files listed in `../reference/updated_metadata.csv` from the `PCD_Spatial` project store into the hisepy cache.

In [ ]:
import os
from pathlib import Path

import hisepy as hp
import pandas as pd

METADATA_PATH = Path('../reference/updated_metadata.csv')

In [ ]:
metadata = pd.read_csv(METADATA_PATH)
wanted_basenames = [os.path.basename(path) for path in metadata.loc[metadata['flag'].isna(), 'file_path']]

store = hp.list_files_in_project_store('PCD_Spatial')
store['basename'] = store['name'].fillna('').map(os.path.basename)
matched = store[store['basename'].isin(wanted_basenames)]

assert not matched['basename'].duplicated().any(), 'Duplicate basenames in the store for wanted files'
missing = sorted(set(wanted_basenames) - set(matched['basename']))
assert not missing, f'Wanted files not found in the store: {missing}'

In [ ]:
downloaded_paths = hp.cache_files(file_ids=matched['id'].tolist())
assert len(downloaded_paths) == len(matched), f'{len(downloaded_paths)} downloaded, {len(matched)} requested'